# Prompt Piper — small LLM experiment (GPT-2 + HellaSwag)

This notebook runs a real end-to-end experiment on a **laptop CPU** using
the smallest viable setup:

* **Model**: `gpt2` (~500 MB, no GPU needed)
* **Task**: `hellaswag` (4-way multiple-choice, loglikelihood scoring)
* **Dataset**: 30 validation samples for search, 50 for final test
* **Search**: NSGA-II, pop=8, 15 generations — completes in a few minutes

Before scaling to a server with a larger model, use this notebook to:
1. Check the pipeline works end-to-end
2. Get a feel for which operators are useful on HellaSwag
3. Understand the Pareto trade-off between token reduction and accuracy

**Sections:**
1. Load model and dataset
2. Inspect the data
3. Baseline accuracy
4. Run the optimisation
5. Inspect the Pareto front
6. Apply best pipeline to a sample
7. Template-only compression mode
8. Operator frequency analysis

In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 120)

## 1. Load model and dataset

`LMEvalBackend.build` loads the HuggingFace model and the lm-eval task once.
Both the evaluator and (optionally) the semantic-compression backend share the same model object.

**Notes for laptop use:**
- `device="cpu"` — no GPU required; avoid `dtype="float16"` on CPU
- GPT-2 cannot follow instructions, so we keep `use_llm_for_semantic=False`;
  semantic operators O23–O28 will use their heuristic fallbacks
- HellaSwag has no public test labels; always use `split="validation"`

In [ ]:
from prompt_piper.backends.lm_eval import LMEvalBackend

backend = LMEvalBackend.build(
    model_type="hf",
    model_name="gpt2",
    task_name="hellaswag",
    device="cpu",        # explicit CPU; remove for GPU
)

print("Model and task loaded.")
print(f"Task output type: {getattr(backend.task, 'OUTPUT_TYPE', 'unknown')}")

## 2. Inspect the data

Build a small dataset from the validation split.  Each sample contains:

| key | content |
|---|---|
| `"prompt"` | Full few-shot formatted prompt (with any prepended examples) |
| `"prompt_template"` | 0-shot version of the same prompt (just the stem + choices) |
| `"target"` | Gold label (int index into the four choices) |
| `"_doc"` | Raw lm-eval document — needed internally by the evaluator |

In [ ]:
# Build a slightly larger pool for browsing; we'll use a subset for search.
# num_fewshot=0 keeps prompts short and fast to evaluate on CPU.
dataset = backend.get_dataset(
    split="validation",
    num_fewshot=0,
    limit=100,
    prompt_field="prompt",
    template_field="prompt_template",   # 0-shot version stored separately
).as_list()

print(f"Loaded {len(dataset)} samples")
print(f"Keys in each sample: {list(dataset[0].keys())}")
print()
print(f"Token counts (whitespace) — first 10 samples:")
for s in dataset[:10]:
    n_full = len(s["prompt"].split())
    n_tmpl = len(s["prompt_template"].split())
    print(f"  full={n_full:4d}  template={n_tmpl:4d}  target={s['target']!r}")

In [ ]:
# Print one full sample so you can see the HellaSwag prompt structure.
sample = dataset[0]

print("=== Full prompt ===")
print(sample["prompt"])
print()
print("=== Template (0-shot) ===")
print(sample["prompt_template"])
print()
print(f"=== Target: {sample['target']!r} ===")

## 3. Baseline accuracy

Before running the search, check how well GPT-2 performs on uncompressed prompts.
This sets the upper bound for what compression can reasonably preserve.

We evaluate on the first 20 samples — enough for a rough estimate without waiting too long.

In [ ]:
# Measure raw accuracy on the first 20 samples (uncompressed prompts).
# Each call runs 4 loglikelihood passes through GPT-2 (one per HellaSwag choice).
# Expect ~30-60 seconds on a modern laptop CPU.

baseline_samples = dataset[:20]
scores = []

for i, s in enumerate(baseline_samples):
    score = backend.evaluator.evaluate(s["prompt"], s["prompt"], s)
    scores.append(score)
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(baseline_samples)} — running accuracy: {sum(scores)/len(scores):.3f}")

print(f"\nBaseline accuracy (n=20): {sum(scores)/len(scores):.3f}")
print(f"(GPT-2 HellaSwag chance = 0.25; random = 0.25; SOTA ≈ 0.95)")

## 4. Run the optimisation search

We use a very small budget to keep this runnable on a laptop:

| Setting | Value | Why |
|---|---|---|
| `n_subset` | 20 | samples evaluated per fitness call |
| `pop_size` | 8 | individuals per generation |
| `n_gen` | 15 | generations |
| `use_llm_for_semantic` | `False` | GPT-2 can't follow compression instructions |

Total forward passes ≈ `pop_size × n_gen × n_subset × 4 choices` = 9 600.
On a modern laptop CPU this takes roughly **5–15 minutes** depending on clock speed.
Scale up all three numbers on the server.

In [ ]:
from prompt_piper.optimization import Experiment, ExperimentConfig

config = ExperimentConfig(
    algorithm="nsga2",
    pop_size=8,
    n_gen=15,
    n_subset=20,       # samples used per fitness evaluation
    subset_seed=42,
    seed=42,
)

exp = backend.build_experiment(
    config,
    split="validation",
    num_fewshot=0,
    use_llm_for_semantic=False,   # GPT-2 heuristic fallbacks only
)

result = exp.run(baselines=["no_compression", "all_transformations", "random_search"])
print(result.summary())

## 5. Inspect the Pareto front

Each Pareto solution is a binary mask over the 28 operators.  Points further left
are more compressed; points higher up preserve more accuracy.

In [ ]:
from prompt_piper.transformations import ALL_TRANSFORMATIONS
import numpy as np

rows = []
for bits, names, tokens, perf in zip(
    result.pareto_solutions,
    result.pareto_transformation_names,
    result.avg_tokens,
    result.avg_performance,
):
    rows.append({
        "avg_tokens":     round(float(tokens), 1),
        "accuracy":       round(float(perf), 3),
        "n_ops":          int(bits.sum()),
        "operators":      ", ".join(names) if names else "(none)",
    })

df_pareto = pd.DataFrame(rows).sort_values("avg_tokens")
df_pareto

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

# Pareto front
idx = result.avg_tokens.argsort()
ax.plot(result.avg_tokens[idx], result.avg_performance[idx],
        "o-", label="NSGA-II Pareto front", zorder=3)

for tokens, perf, bits in zip(result.avg_tokens, result.avg_performance, result.pareto_solutions):
    ax.annotate(f"{int(bits.sum())} ops", (tokens, perf),
                textcoords="offset points", xytext=(4, 4), fontsize=7)

# Baselines
markers = {"no_compression": ("gray", "*", 250),
           "all_transformations": ("red", "D", 80),
           "random_search": ("orange", "^", 100)}
for name, br in result.baselines.items():
    c, m, s = markers.get(name, ("black", "o", 80))
    ax.scatter(br.avg_tokens.mean(), br.avg_performance.mean(),
               color=c, marker=m, s=s, zorder=4, label=name)

ax.set_xlabel("Average tokens (lower = more compressed)")
ax.set_ylabel("Accuracy (HellaSwag)")
ax.set_title("GPT-2 on HellaSwag — Pareto front")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Apply the best-compression pipeline to a sample

The best-compression solution has the fewest tokens; inspect what it actually
does to a HellaSwag prompt.

In [ ]:
bits_comp, tok_comp, perf_comp = result.best_compression()
bits_perf, tok_perf, perf_perf = result.best_performance()

print(f"Best compression : {tok_comp:.1f} tokens, accuracy={perf_comp:.3f}")
print(f"Best performance : {tok_perf:.1f} tokens, accuracy={perf_perf:.3f}")

transforms = [cls() for cls in ALL_TRANSFORMATIONS]

ops_comp = [t.name for t, b in zip(transforms, bits_comp) if b]
ops_perf = [t.name for t, b in zip(transforms, bits_perf) if b]

print(f"\nBest-compression operators ({len(ops_comp)}):")
for name in ops_comp:
    print(f"  {name}")

print(f"\nBest-performance operators ({len(ops_perf)}):")
for name in ops_perf:
    print(f"  {name}")

In [ ]:
# Apply best-compression pipeline to the first sample and compare.
pipeline = [t for t, b in zip(transforms, bits_comp) if b]

original  = dataset[0]["prompt"]
compressed = original
for op in pipeline:
    compressed = op(compressed)

print(f"Original  ({len(original.split())} words):")
print(original)
print()
print("─" * 70)
print(f"Compressed ({len(compressed.split())} words):")
print(compressed)

In [ ]:
# Score the compressed prompt on the first sample with the real evaluator.
s = dataset[2]
score_original   = backend.evaluator.evaluate(s["prompt"], s["prompt"],   s)
score_compressed = backend.evaluator.evaluate(s["prompt"], compressed, s)

print(f"Original   accuracy on sample 0: {score_original:.1f}")
print(f"Compressed accuracy on sample 0: {score_compressed:.1f}")
print(f"Tokens: {len(s['prompt'].split())} → {len(compressed.split())} "
      f"({len(compressed.split())/len(s['prompt'].split()):.1%} of original)")

## 7. Template-only compression

With `compress_field="prompt_template"` the search compresses the **0-shot template** only —
excluding any few-shot examples (here there are none, but this matters when `num_fewshot > 0`
on the server).

Compare the two Pareto fronts side by side.

In [ ]:
config_tmpl = ExperimentConfig(
    algorithm="nsga2",
    pop_size=8,
    n_gen=15,
    n_subset=20,
    subset_seed=42,
    seed=42,
    compress_field="prompt_template",   # compress only the 0-shot part
)

exp_tmpl = backend.build_experiment(
    config_tmpl,
    split="validation",
    num_fewshot=0,
    use_llm_for_semantic=False,
)

result_tmpl = exp_tmpl.run(baselines=["no_compression", "all_transformations"])
print(result_tmpl.summary())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for res, label, color in [
    (result,      "Full-prompt compression",   "steelblue"),
    (result_tmpl, "Template-only compression", "darkorange"),
]:
    idx = res.avg_tokens.argsort()
    ax.plot(res.avg_tokens[idx], res.avg_performance[idx],
            "o-", label=label, color=color)

# No-compression baseline from the full-prompt run
nc = result.baselines["no_compression"]
ax.scatter(nc.avg_tokens.mean(), nc.avg_performance.mean(),
           marker="*", s=250, color="gray", zorder=5, label="no_compression")

ax.set_xlabel("Average tokens in the compressed field")
ax.set_ylabel("Accuracy (HellaSwag)")
ax.set_title("Full-prompt vs template-only compression")
ax.legend()
plt.tight_layout()
plt.show()

for label, res in [("Full-prompt", result), ("Template", result_tmpl)]:
    _, tok, perf = res.best_compression()
    print(f"{label:14s}  best compression: {tok:.1f} tok / {perf:.3f} acc")

## 8. Operator frequency analysis

Which operators appear most frequently on the Pareto front?
Operators that recur across many solutions are robust choices for HellaSwag —
they compress reliably without hurting loglikelihood scoring.

In [ ]:
from collections import Counter

counts = Counter()
for names in result.pareto_transformation_names:
    counts.update(names)

if counts:
    ops, freqs = zip(*counts.most_common())
    fig, ax = plt.subplots(figsize=(8, max(3, len(ops) * 0.35)))
    ax.barh(list(ops)[::-1], list(freqs)[::-1])
    ax.set_xlabel("Frequency on Pareto front")
    ax.set_title("Most common operators — GPT-2 / HellaSwag")
    plt.tight_layout()
    plt.show()
else:
    print("No operators selected on the Pareto front.")

## Next steps on the server

When moving to a GPU server, scale up by adjusting a few parameters:

```python
from prompt_piper.backends.lm_eval import LMEvalBackend
from prompt_piper.optimization import ExperimentConfig

backend = LMEvalBackend.build(
    model_type="hf",
    model_name="mistralai/Mistral-7B-v0.1",   # larger model
    task_name="hellaswag",
    device="cuda",
    dtype="float16",
)

config = ExperimentConfig(
    algorithm="nsga2",
    pop_size=50,          # was 8
    n_gen=100,            # was 15
    n_subset=200,         # was 20
    compress_field="prompt_template",
)

result = backend.build_experiment(
    config,
    split="validation",
    num_fewshot=5,        # add few-shot examples
    use_llm_for_semantic=True,   # enable LLM-backed O23–O28
).run(baselines=["no_compression", "all_transformations", "random_search"])

print(result.summary())
```